# CYTools-agent

An agent that drives CYTools (fetch polytopes, triangulate, build CYs) with a local Ollama model.

Run with the **Python (cytools-agent)** kernel. Run the **Setup** cells once, then **chat** — add `agent.chat(...)` cells freely; the agent remembers earlier turns. Re-run the *Start a session* cell to reset.

## Setup (run once)

In [ ]:
from openai import OpenAI
import os

base = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
client = OpenAI(base_url=base + "/v1", api_key="ollama")
MODEL = "qwen3:8b"
assert MODEL in [m.id for m in client.models.list().data], f"{MODEL} not pulled"
print("OK:", MODEL)

In [ ]:
from cytools_agent.tools import (polytope, triangulation, cy, code)
from cytools_agent.schema import function_to_schema

TOOL_FNS = [polytope.fetch_polytopes, polytope.get_polytope_info,
            polytope.ks_stats,
            triangulation.get_heights,
            triangulation.get_triangulation_info,
            cy.get_cy_info, cy.get_cy_cones,
            code.run_python, code.cytools_help]
tools = [function_to_schema(fn) for fn in TOOL_FNS]
tool_impls = {fn.__name__: fn for fn in TOOL_FNS}

In [ ]:
from cytools_agent.agent import Agent
from cytools_agent.prompt import DEFAULT_SYSTEM_PROMPT as system_prompt
# (edit system_prompt here to customize)

## Start a session

Run this to begin — or re-run it to reset the conversation.

In [ ]:
agent = Agent(client, MODEL, system_prompt, tools, tool_impls,
              max_steps=20, verbosity=2)

## Chat

Add more `agent.chat(...)` cells below; the agent remembers earlier turns.

In [ ]:
print(agent.chat("Fetch 3 polytopes at h11=5"))

In [ ]:
print(agent.chat("How many NTFEs do each of them have?"))

In [ ]:
print(agent.chat("What are the CY volumes at the tip of the stretched kahler cone for each of their associated CYs?"))

In [ ]:
# Save the session as a standalone Python script:
# agent.save_history("session.py")

## Orchestrator (PM + engineer)

For bigger multi-step questions (loop over many polytopes, compute, plot), use the **orchestrator** instead of `agent.chat`: a project-manager model plans the work and a separate engineer executes it step by step, with every result captured in an evidence log.

One call does everything — no setup beyond the imports:

```python
from cytools_agent.orchestrator import run_session
print(run_session("your question", model=MODEL))
```

**Watch it live:** run `python -m cytools_agent.viewer` in a terminal and open http://127.0.0.1:8765 — the plan, each engineer step (code + real output), and figures render as the session runs. Finished sessions are archived in `scratch/logs/` and browsable in the same viewer; `python -m cytools_agent.viewer export` bakes one into a shareable standalone HTML.

In [ ]:
from cytools_agent.orchestrator import run_session

print(run_session(
    "Among the first 500 polytopes at h11=8, plot the distribution of the "
    "number of NTFE triangulations, and report the mean.",
    model=MODEL))

In [ ]:
print(run_session(
    "Fetch the first 10x polytopes at each h11 in [2,10] and "
    "plot the size of their automorphism group vs h11",
    model=MODEL))

In [ ]:
# For research questions where you want extra confidence: run several
# independent sessions and accept the answer only when the final numbers
# agree (~votes x the wall clock; disagreement is flagged LOW CONFIDENCE).
# from cytools_agent.orchestrator import run_session_voted
# print(run_session_voted("your question", votes=3, model=MODEL))

### Conversational orchestrator

`OrchestratorChat` keeps state between questions: the polytopes and columns one turn computes stay available, so follow-ups like *"those same polytopes"* or *"now plot that vs h21"* just work. Plots can sweep h11 ranges, color points by a third quantity, overlay histograms by category, and use log axes — and you can ask for several figures in one question.

In [ ]:
from cytools_agent.orchestrator import OrchestratorChat

ochat = OrchestratorChat(model=MODEL, verbose=False)
print(ochat.chat("Fetch the first 25 polytopes at h11=3 and report how many "
                 "NTFE triangulations each has."))

In [ ]:
print(ochat.chat("Now scatter those NTFE counts against the polytopes' h21 "
                 "values."))